# 00 — Data Validation

Validates schema, ranges, keys, provenance, collection coverage, and submission readiness. Raw inputs are never modified.

In [ ]:
from pathlib import Path
import json
import warnings
import numpy as np
import pandas as pd
import yaml

warnings.filterwarnings("ignore")
current = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in (current, *current.parents) if (p / "config" / "config.yaml").exists()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Cannot locate project root containing config/config.yaml")
with (PROJECT_ROOT / "config" / "config.yaml").open(encoding="utf-8") as handle:
    CONFIG = yaml.safe_load(handle)

def project_path(key):
    return PROJECT_ROOT / CONFIG["paths"][key]

OUTPUT = project_path("output")
PROCESSED = project_path("processed")
for folder in [OUTPUT / "reports", OUTPUT / "figures", OUTPUT / "models", OUTPUT / "labeling", OUTPUT / "logs", OUTPUT / "powerbi", PROCESSED]:
    folder.mkdir(parents=True, exist_ok=True)
print("PROJECT_ROOT:", PROJECT_ROOT)

In [ ]:
SHOP_REQUIRED = ["Shop_ID", "Shop_Name", "Shop_type", "Years_Active", "Is_Online_Now", "Total_Products", "Follower_Count_k", "Rating_Average", "Total_Ratings_k", "Chat_Response_Rate_pct", "Has_Voucher", "Target_Label", "Time_Collected"]
PRODUCT_REQUIRED = ["Shop_ID", "product_id", "product_name", "product_details", "description_text", "Review_stars", "units_sold", "product_url", "Time_Collected", "Data_Source"]
REVIEW_REQUIRED = ["Shop_ID", "product_id", "review_id", "user_name", "rating", "review_time", "review_text", "has_image", "Data_Source"]
SELLER_REQUIRED = ["Shop_ID", "product_id", "Conversion_Rate_pct", "Time_Collected", "Data_Source"]

def load_json_folder(folder, pattern, columns):
    records = []
    for path in sorted(folder.glob(pattern)):
        with path.open(encoding="utf-8") as handle:
            payload = json.load(handle)
        if not isinstance(payload, list):
            raise ValueError(f"{path}: top-level JSON must be a list")
        records.extend(payload)
    return pd.DataFrame(records) if records else pd.DataFrame(columns=columns)

def load_seller_metrics(path):
    candidates = [path, path.with_suffix(".xlsx")]
    source = next((p for p in candidates if p.exists()), None)
    if source is None:
        return pd.DataFrame(columns=SELLER_REQUIRED), None
    frame = pd.read_excel(source, dtype={"Shop_ID": str, "product_id": str}) if source.suffix == ".xlsx" else pd.read_csv(source, dtype={"Shop_ID": str, "product_id": str})
    return frame, source

shops = pd.read_excel(project_path("shops"), dtype={"Shop_ID": str})
products = load_json_folder(project_path("products"), "shop_*_products.json", PRODUCT_REQUIRED)
reviews = load_json_folder(project_path("reviews"), "shop_*_reviews.json", REVIEW_REQUIRED)
seller_metrics, seller_source = load_seller_metrics(project_path("seller_metrics"))

# Canonicalize legacy/manual collection fields without inventing business values.
manifest_path = PROJECT_ROOT / "src" / "crawl_data" / "product_review_mapping.json"
if manifest_path.exists():
    manifest = pd.DataFrame(json.loads(manifest_path.read_text(encoding="utf-8")))
    if {"product_id", "product_url"} <= set(manifest):
        url_map = manifest.drop_duplicates("product_id").set_index("product_id")["product_url"]
        if "product_url" not in products: products["product_url"] = products["product_id"].map(url_map)
        else: products["product_url"] = products["product_url"].fillna(products["product_id"].map(url_map))
shop_dates = shops.set_index("Shop_ID")["Time_Collected"].to_dict() if {"Shop_ID", "Time_Collected"} <= set(shops) else {}
for column in PRODUCT_REQUIRED:
    if column not in products: products[column] = np.nan
products["Time_Collected"] = products["Time_Collected"].fillna(products["Shop_ID"].map(shop_dates))
products["Data_Source"] = products["Data_Source"].fillna("Shopee public listing (legacy/manual collection)")
for column in REVIEW_REQUIRED:
    if column not in reviews: reviews[column] = np.nan
if "collection_date" in reviews: reviews["review_time"] = reviews["review_time"].fillna(reviews["collection_date"])
if "source_url" in reviews: reviews["Data_Source"] = reviews["Data_Source"].fillna(reviews["source_url"])
print({"shops": len(shops), "products": len(products), "reviews": len(reviews), "seller_metrics": len(seller_metrics)})

In [ ]:
checks = []
def check(name, severity, invalid_count, description):
    invalid_count = int(invalid_count)
    checks.append({"check_name": name, "severity": severity, "status": "PASS" if invalid_count == 0 else severity, "error_count": invalid_count, "description": description})

def missing_count(frame, required):
    return len(set(required) - set(frame.columns))

for name, frame, required in [("shops", shops, SHOP_REQUIRED), ("products", products, PRODUCT_REQUIRED), ("reviews", reviews, REVIEW_REQUIRED), ("seller_metrics", seller_metrics, SELLER_REQUIRED)]:
    check(f"{name}_required_columns", "FAIL", missing_count(frame, required), f"Required schema: {required}")

if not missing_count(shops, SHOP_REQUIRED):
    check("unique_Shop_ID", "FAIL", shops["Shop_ID"].duplicated().sum(), "Shop_ID must be unique")
    check("shop_rating_range", "FAIL", (~pd.to_numeric(shops["Rating_Average"], errors="coerce").between(0, 5)).sum(), "Rating_Average in [0,5]")
    check("response_rate_range", "FAIL", (~pd.to_numeric(shops["Chat_Response_Rate_pct"], errors="coerce").between(0, 100)).sum(), "Chat response in [0,100]")
    check("shop_dates", "FAIL", pd.to_datetime(shops["Time_Collected"], errors="coerce").isna().sum(), "Time_Collected must be a date")
if not missing_count(products, PRODUCT_REQUIRED):
    check("unique_product_id", "FAIL", products["product_id"].duplicated().sum(), "product_id must be globally unique")
    product_rating = pd.to_numeric(products["Review_stars"], errors="coerce")
    product_sales = pd.to_numeric(products["units_sold"], errors="coerce")
    check("product_rating_range", "FAIL", (product_rating.notna() & ~product_rating.between(0, 5)).sum(), "Observed Review_stars in [0,5]")
    check("product_rating_missing", "WARNING", product_rating.isna().sum(), "Missing rating remains null")
    check("product_units_sold_negative", "FAIL", (product_sales.notna() & (product_sales < 0)).sum(), "Observed units_sold must be non-negative")
    check("product_units_sold_missing", "WARNING", product_sales.isna().sum(), "Missing sales remains null; never substitute ratings")
    check("product_provenance", "WARNING", products[["product_url", "Time_Collected", "Data_Source"]].isna().any(axis=1).sum(), "Products should have URL, date and source")
if not missing_count(reviews, REVIEW_REQUIRED):
    check("unique_review_id", "FAIL", reviews["review_id"].duplicated().sum(), "review_id must be unique")
    review_rating = pd.to_numeric(reviews["rating"], errors="coerce")
    review_image = pd.to_numeric(reviews["has_image"], errors="coerce")
    check("review_rating_range", "FAIL", (review_rating.notna() & ~review_rating.between(1, 5)).sum(), "Observed rating in [1,5]")
    check("review_rating_missing", "WARNING", review_rating.isna().sum(), "Unverified rating remains null")
    check("review_has_image", "FAIL", (review_image.notna() & ~review_image.isin([0, 1])).sum(), "Observed has_image must be binary")
    check("review_has_image_missing", "WARNING", review_image.isna().sum(), "Unverified image flag remains null")
    check("review_text", "FAIL", reviews["review_text"].fillna("").astype(str).str.strip().eq("").sum(), "Only text reviews are accepted")
if not missing_count(seller_metrics, SELLER_REQUIRED):
    check("seller_key_unique", "FAIL", seller_metrics.duplicated(["Shop_ID", "product_id"]).sum(), "Seller export must have one row per shop/product")
    check("conversion_range", "FAIL", (~pd.to_numeric(seller_metrics["Conversion_Rate_pct"], errors="coerce").between(0,100)).sum(), "Conversion rate in [0,100]")

if {"Shop_ID"} <= set(shops) and {"Shop_ID", "product_id"} <= set(products) and {"Shop_ID", "product_id"} <= set(reviews):
    check("product_shop_fk", "FAIL", len(set(products["Shop_ID"]) - set(shops["Shop_ID"])), "Every product shop exists")
    check("review_shop_fk", "FAIL", len(set(reviews["Shop_ID"]) - set(shops["Shop_ID"])), "Every review shop exists")
    check("review_product_fk", "FAIL", len(set(reviews["product_id"]) - set(products["product_id"])), "Every review product exists")
    product_shop = products.set_index("product_id")["Shop_ID"].to_dict()
    check("review_product_shop_match", "FAIL", reviews.apply(lambda r: product_shop.get(r["product_id"]) not in (None, r["Shop_ID"]), axis=1).sum(), "Review Shop_ID agrees with its product")

dq = CONFIG["data_quality"]
check("minimum_shops", "WARNING", max(0, dq["minimum_shops"] - shops["Shop_ID"].nunique()), f"At least {dq['minimum_shops']} real shops")
product_counts = products.groupby("Shop_ID").size().reindex(shops["Shop_ID"], fill_value=0)
review_counts = reviews.groupby(["Shop_ID", "product_id"]).size().reindex(pd.MultiIndex.from_frame(products[["Shop_ID", "product_id"]]), fill_value=0)
check("products_per_shop", "WARNING", ((product_counts < dq["minimum_products_per_shop"]) | (product_counts > dq["maximum_products_per_shop"])).sum(), "Each shop needs 5–10 products")
check("reviews_per_product", "WARNING", (review_counts < dq["minimum_text_reviews_per_product"]).sum(), "Each product needs at least five text reviews")
check("seller_metrics_present", "WARNING", int(seller_metrics.empty), "Authorized Seller Centre export is required")

coverage = pd.DataFrame({"Shop_ID": shops["Shop_ID"], "Product_Count": shops["Shop_ID"].map(product_counts).fillna(0).astype(int)})
coverage["Review_Count"] = coverage["Shop_ID"].map(reviews.groupby("Shop_ID").size()).fillna(0).astype(int)
coverage["Products_With_Conversion"] = coverage["Shop_ID"].map(seller_metrics.groupby("Shop_ID").size() if not seller_metrics.empty else {}).fillna(0).astype(int)
report = pd.DataFrame(checks)
report.to_csv(OUTPUT / "reports" / "data_validation_report.csv", index=False)
coverage.to_csv(OUTPUT / "reports" / "data_coverage_by_shop.csv", index=False)
display(report); display(coverage)
critical = report.query("status == 'FAIL'")
if not critical.empty:
    raise ValueError("Critical validation failed: " + ", ".join(critical["check_name"]))

In [ ]:
def exists(rel): return (PROJECT_ROOT / rel).exists()
items = {
    "THREE_DATA_TYPES": bool(len(shops) and len(products) and len(reviews)),
    "MINIMUM_SHOPS": shops["Shop_ID"].nunique() >= CONFIG["data_quality"]["minimum_shops"],
    "SELLER_METRICS": not seller_metrics.empty,
    "POWER_BI": any((PROJECT_ROOT / "powerbi").glob("*.pbix")),
    "CLUSTERING_NOTEBOOK": exists("src/03_clustering_reviews.ipynb"),
    "CLASSIFICATION_NOTEBOOK": exists("src/04_shop_classification.ipynb"),
    "REGRESSION_NOTEBOOK": exists("src/05_regression.ipynb"),
    "AUTHENTICITY_REPORT": exists("output/reports/Shop_Authenticity_Report.csv"),
    "CLASSIFICATION_RESULT": exists("output/reports/Classification_Result.csv"),
    "REGRESSION_RESULT": exists("output/reports/Regression_Result.csv"),
    "FINAL_REPORT": any((PROJECT_ROOT / "report").glob("Final_Report.*")),
}
text = "\n".join(f"{name:<30} {'PASS' if value else 'FAIL'}" for name, value in items.items())
text += f"\n\nOVERALL: {'READY' if all(items.values()) else 'NOT READY FOR SUBMISSION'}\n"
(OUTPUT / "reports" / "submission_check.txt").write_text(text, encoding="utf-8")
print(text)